In [3]:

"""
关于模型调用的方法：
    为了尽可能简化自定义链的创建，我们实现了一个Runnable协议，许多LangChain组件实现了
    Runnable协议，包括聊天模型，提示词模板，输出解析器，检索器，代理（智能体）等
    Runnable定义的公共调用方法如下：
        invoke：处理单条输入，等待LLM完全推理完成后再返回调用结果
        stream：流式响应，逐字输出LLM的响应结果
        batch：处理批量输入
        这些也有相应的异步方法，应该与asyncio和await语法一起使用以实现并发：
        astream：异步流式响应
        ainvoke：异步处理单条输入
        abatch：异步处理批量输入
        astream_log:异步流式返回中间步骤，以及最终响应
        astream_events:异步流式返回链中发生的事件
"""

from langchain_core.messages import HumanMessage, SystemMessage


from langchain_openai import ChatOpenAI
# 阻塞式
# llm = ChatOpenAI(
#     model_name="GLM-5.1",# 默认使用的是gpt3.5
#     base_url="",
#     api_key="",
# )

# message = [HumanMessage(content='你好，请介绍一下自己')]

# resp = llm.invoke(message)
# print(resp.content)

# 流式

llm = ChatOpenAI(
    model_name="GLM-5.1",# 默认使用的是gpt3.5
    base_url="",
    api_key="",
    streaming=True
)

message = [HumanMessage(content='你好，请介绍一下自己')]

print('开始流式输出')

for chunk in llm.stream(message):
    print(chunk.content,end='',flush=True)

print('\n流式输出结果')


开始流式输出
你好！我是GLM，由Z.ai开发的大语言模型。我通过大规模文本数据训练，能够理解和生成自然语言内容。

我的设计目标是帮助用户解答问题、提供信息和完成各种语言任务，同时持续学习和优化自己的能力。我不会存储您的个人数据，请放心交流。

有什么我能帮助你解答的问题或完成的任务吗？
流式输出结果


In [4]:
# 批量调用

from langchain_core.messages import HumanMessage, SystemMessage


from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model_name="GLM-5.1",# 默认使用的是gpt3.5
    base_url="",
    api_key=""
)

message_1 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='你好，请介绍一下自己')]
message_2 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='强帮我介绍一下什么是AIGC')]
message_3 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='你好，请帮我介绍一下什么式大模型技术')]

message = [message_1,message_2,message_3]

resp = llm.batch(message)

print(resp)


[AIMessage(content='你好！很高兴认识你！😊\n\n我是一位乐于助人的AI小助手。我的目标就是为你提供便捷、高效的支持和陪伴，让你的生活和工作变得更轻松。\n\n我可以做很多事情，比如：\n* **回答问题**：无论是科学知识、历史事件还是生活小常识，我都可以尽力为你解答。\n* **文案创作**：帮你写邮件、总结文章、构思故事、写报告或是生成创意文案。\n* **语言翻译**：支持多种语言之间的互译，帮你打破语言障碍。\n* **逻辑与编程**：帮你梳理思路、分析问题，甚至编写和调试代码。\n* **日常闲聊**：随时倾听你的分享，陪你聊聊天，解解闷。\n\n总之，遇到任何问题或者需要帮忙的时候，随时都可以找我！请问今天有什么我可以为你效劳的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 692, 'prompt_tokens': 18, 'total_tokens': 710, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 519, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'GLM-5.1', 'system_fingerprint': None, 'id': '20260604183250f7c7fef1d2774b19', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--c78b7cab-acb6-42e0-a143-1c4c765915ae-0', usage_metadata={'input_tokens': 18, 'output_tokens': 692, 'total_tokens': 710, 'input_token

In [5]:
import time

# 关于同步和异步

def call_model():
    print('开始调用模型...')
    time.sleep(5)
    print('模型调用结束')

def perform_other_tasks():
    # 模拟执行其它任务
    for i in range(5):
        print(f'执行其它任务{i + 1}')
        time.sleep(1)
def main():
    start_time = time.time()
    call_model()
    perform_other_tasks()
    end_time = time.time()
    total_time = end_time - start_time
    return f'耗时{total_time}秒'

main_time = main()
print(main_time)

开始调用模型...
模型调用结束
执行其它任务1
执行其它任务2
执行其它任务3
执行其它任务4
执行其它任务5
耗时10.00338888168335秒


In [11]:
import asyncio

async def call_model():
    print('开始调用模型...')
    await asyncio.sleep(5)
    print('模型调用结束')

async def perform_other_tasks():
    # 模拟执行其它任务
    for i in range(5):
        print(f'执行其它任务{i + 1}')
        await asyncio.sleep(1)
async def main():
    start_time = time.time()
    # 并发执行两个任务
    await asyncio.gather(
        call_model(),
        perform_other_tasks()
    )
    end_time = time.time()
    total_time = end_time - start_time
    return f'耗时{total_time}秒'

main_time = await main()
print(main_time)

开始调用模型...
执行其它任务1
执行其它任务2
执行其它任务3
执行其它任务4
执行其它任务5
模型调用结束
耗时5.057931900024414秒


In [2]:

from langchain_core.messages import HumanMessage, SystemMessage


from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model_name="GLM-5.1",# 默认使用的是gpt3.5
    base_url="",
    api_key=""
)

message_1 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='你好，请介绍一下自己')]
message_2 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='强帮我介绍一下什么是AIGC')]
message_3 = [SystemMessage(content='你是以为乐于助人的小助手'),HumanMessage(content='你好，请帮我介绍一下什么式大模型技术')]

# resp_1 = llm.invoke(message_1)
# resp_2 = llm.invoke(message_2)
# resp_3 = llm.invoke(message_3)

resp_1, resp_2, resp_3 = await asyncio.gather(
    llm.ainvoke(message_1),
    llm.ainvoke(message_2),
    llm.ainvoke(message_3)
)
print(resp_1)
print(resp_2)
print(resp_3)

RateLimitError: Error code: 429 - {'error': {'code': '1113', 'message': '余额不足或无可用资源包,请充值。'}}